# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [02:05<00:00, 25.12s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

'Title: eBay Promo Code: Extra 20% off + free shipping\nDetails: The promo code "LOVETOSAVE" cuts an extra 20% off over 5,000 items in eBay\'s Brand Outlet section. It applies to Crocs, adidas, Bose, Dyson, and lots of other big-brand products, yielding some of the best prices we\'ve seen in months. A maximum discount of $500 applies and the coupon code can be used twice per account. Shop Now at eBay\nFeatures: \nURL: https://www.dealnews.com/eBay-Promo-Code-Extra-20-off-free-shipping/21770080.html?iref=rss-c196'

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Apple Watch Ultra 2 GPS + Cellular 49mm Smartwatch for $649 + free shipping
Details: That's the second best price we've seen and a low today by about $150 for a new one. Buy Now at Best Buy
Features: up to 36 hours of battery life fall & crash detection Model: MX4D3LW/A
URL: https://www.dealnews.com/products/Apple/Apple-Watch-Ultra-2-GPS-Cellular-49-mm-Smartwatch/49

In [9]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [10]:
result = get_recommendations()

In [11]:
len(result.deals)

5

In [12]:
result.deals[1]

Deal(product_description='The Garmin Fenix 7X Sapphire Solar GPS Smartwatch is a top-notch multisport GPS watch, perfect for fitness enthusiasts and outdoor adventurers alike. It showcases a 1.4” solar-powered display and boasts an impressive 28 days of battery life in smartwatch mode. This smartwatch comes equipped with multi-band GPS and features for 24/7 heart rate and sleep tracking. It also includes maps for global navigation, making it indispensable for hikers and runners. The durability of the sapphire glass adds to its rugged appeal.', price=500.0, url='https://www.dealnews.com/products/Garmin/Garmin-Fenix-7-X-Sapphire-Solar-GPS-Smartwatch/493064.html?iref=rss-c142')

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

In [15]:
result

DealSelection(deals=[Deal(product_description='The Apple Watch Ultra 2 GPS + Cellular Smartwatch offers advanced features for fitness and connectivity. With a stunning 49mm case, this watch is equipped with a battery life of up to 36 hours, ensuring you stay powered throughout the day. It includes fall and crash detection capabilities, making it a reliable companion for outdoor activities. This premium smartwatch combines durability with smart technology, making it ideal for both casual and serious users.', price=649.0, url='https://www.dealnews.com/products/Apple/Apple-Watch-Ultra-2-GPS-Cellular-49-mm-Smartwatch/493696.html?iref=rss-c142'), Deal(product_description='The Garmin Fenix 7X Sapphire Solar GPS Smartwatch is designed for outdoor enthusiasts. Featuring a 1.4” solar-powered display, it boasts an impressive 28 days of battery life in smartwatch mode. This device provides multi-band GPS, GLONASS, and Galileo support, making it perfect for precise navigation in various terrains. 